<a href="https://colab.research.google.com/github/Biondi-Tommaso/EarthObservation/blob/main/UHI_GE_Biondi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisi Isole di Calore Urbano (UHI) - Genova

Obiettivo: Analisi delle isole di caolore nella città di Genova

## 1\. Setup e Librerie

Installa e importa le librerie necessarie e inizializza Earth Engine.

In [ ]:
try:
    import geemap
    import ee
except ImportError:
    !pip install geemap
    import geemap
    import ee

import json
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# Autenticazione e Inizializzazione
try:
    ee.Initialize(project='gmail-361617') # <- inserite il vostro qui
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

print("Librerie importate e Earth Engine inizializzato.")

Librerie importate e Earth Engine inizializzato.


## 2\. Caricamento Area di Interesse (ROI)

Carica il file GeoJSON da Google Drive o definisce una geometria di default se il file non è disponibile.

In [ ]:
from google.colab import drive
import os

# Monta Drive
drive.mount('/content/drive') # importo i dati dal drive per rapidita, se volete usare un'altra soluzione commentate questa riga e modificate l'url di sotto

# Percorso del file
geojson_file_path = '/content/drive/MyDrive/Righi.geojson'

def get_roi_from_geojson(path):
    try:
        with open(path) as f:
            geojson_data = json.load(f)
        # Estraggo i dati e rimuovo i dati relativi all'altitudine (che poi riprendo in futuro, ma almeno siamo allineati)
        coords = geojson_data['features'][0]['geometry']['coordinates']

        def remove_altitude(coordinates_list):
            if isinstance(coordinates_list[0], (float, int)):
                return coordinates_list[:2]
            return [remove_altitude(x) for x in coordinates_list]

        cleaned_coords = remove_altitude(coords)

        # Creazione geometria EE
        if geojson_data['features'][0]['geometry']['type'] == 'Polygon':
            return ee.Geometry.Polygon(cleaned_coords)
        elif geojson_data['features'][0]['geometry']['type'] == 'MultiPolygon':
            return ee.Geometry.MultiPolygon(cleaned_coords)

    except Exception as e:
        print(f"Errore nel caricamento del GeoJSON: {e}")
        print("Utilizzo geometria di fallback (centro di Genova).")
        return ee.Geometry.Point([8.934, 44.405]).buffer(5000).bounds()

area = get_roi_from_geojson(geojson_file_path)
print("Geometria di Genova caricata.")

Mounted at /content/drive
Errore nel caricamento del GeoJSON: [Errno 2] No such file or directory: '/content/drive/MyDrive/Righi.geojson'
Utilizzo geometria di fallback (centro di Genova).
Geometria di Genova caricata.


## 3\. Definizione delle Funzioni di Analisi

Qui definiamo tutte le funzioni per il calcolo delle maschere, NDVI, temperatura e normalizzazione topografica.

**Nota sulla Normalizzazione:** Invece di usare un'altitudine costante, qui utilizziamo il **DEM (Digital Elevation Model)**. Questo è cruciale per Genova: permette di capire se una zona è fresca perché è "verde" o semplicemente perché è in collina.

In [ ]:

# 1. Maschera Nuvole
def mask_clouds(image):
    qa_pixel = image.select('QA_PIXEL')
    # Bit 3: Cloud, Bit 5: Cloud Shadow
    mask = qa_pixel.bitwiseAnd(1 << 3).eq(0).And(qa_pixel.bitwiseAnd(1 << 5).eq(0))
    return image.updateMask(mask)

# 2. Calcolo NDVI
def add_ndvi(image):
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return image.addBands(ndvi)

# 3. Calcolo Temperatura Celsius
def add_celsius(image):
    temp_kelvin = image.select('ST_B10').multiply(0.00341802).add(149.0)
    temp_celsius = temp_kelvin.subtract(273.15).rename('LST_Celsius')
    return image.addBands(temp_celsius)

# 4. Normalizzazione Temperatura
def normalize_temperature(image, roi, lapse_rate=0.0065):
    dem = ee.Image('USGS/SRTMGL1_003').clip(roi) #carico l'altitudine

    lst = image.select('LST_Celsius')

    # Correzione: aggiungiamo gradi in base all'altezza (per annullare l'effetto raffreddante della collina)
    correction = dem.multiply(lapse_rate)
    lst_normalized = lst.add(correction).rename('LST_Normalized')

    return image.addBands(lst_normalized)

print("Funzioni definite.")

Funzioni definite.


## 4\. Estrazione ed Elaborazione Dati

Questo blocco itera attraverso gli anni e prepara i dati per la visualizzazione e il grafico, popolando un DataFrame.

In [ ]:
years = range(2013, 2026)
months = range(6, 9)

data_list = []
image_collection_dict = {}


#estraggo solo i dati nell'intervallo interessato
for year in years:
    image_collection_dict[year] = {}
    collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(area)
                  .filterDate(f'{year}-06-01', f'{year}-08-31')
                  .map(mask_clouds)
                  .map(add_ndvi)
                  .map(add_celsius))

    for month in months:
        # Filtra per mese specifico
        monthly_img = collection.filter(ee.Filter.calendarRange(month, month, 'month')).mean()

        # Applica normalizzazione
        monthly_img = normalize_temperature(monthly_img, area)
        monthly_img = monthly_img.clip(area)

        image_collection_dict[year][month] = monthly_img

        try:
            stats = monthly_img.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=area,
                scale=100,
                maxPixels=1e9
            ).getInfo()

            if stats.get('LST_Celsius') is not None:
                data_list.append({
                    'date': pd.to_datetime(f'{year}-{month}-01'),
                    'year': year,
                    'month': month,
                    'LST_Celsius': stats.get('LST_Celsius'),
                    'LST_Normalized': stats.get('LST_Normalized'),
                    'NDVI': stats.get('NDVI')
                })
        except Exception as e:
            continue

# Creazione DataFrame
df = pd.DataFrame(data_list)
print(f"Elaborazione completata. Estratti dati per {len(df)} mesi.")

Elaborazione completata. Estratti dati per 39 mesi.


## 5\. Analisi Grafica

Confronto temporale tra Temperatura (Reale e Normalizzata) e Vegetazione.

In [ ]:
# Normalizzazione dati per confronto visivo (scale 0-1)
df['LST_Norm_Scaled'] = (df['LST_Normalized'] - df['LST_Normalized'].min()) / (df['LST_Normalized'].max() - df['LST_Normalized'].min())
df['NDVI_Scaled'] = (df['NDVI'] - df['NDVI'].min()) / (df['NDVI'].max() - df['NDVI'].min())

fig = px.line(df, x='date', y=['LST_Norm_Scaled', 'NDVI_Scaled'],
              title='Confronto Normalizzato: Temperatura (Corretta per altitudine) vs NDVI a Genova',
              labels={'value': 'Valore Normalizzato (0-1)', 'date': 'Data'},
              markers=True)
fig.show()

## 6\. Mappa Interattiva

Esplora visivamente le Isole di Calore.

In [ ]:
import ee
import ipywidgets as widgets
from IPython.display import display, clear_output

Map = geemap.Map(center=[44.405, 8.934], zoom=12)

style = {'description_width': 'initial'}

reference_years = [y for y in years if y != years[-1]]
year_widget = widgets.Dropdown(options=years, description='Anno (Analisi):', style=style, value=years[-1])
month_widget = widgets.Dropdown(options=months, description='Mese:', style=style)
layer_widget = widgets.Dropdown(options=['LST_Celsius', 'LST_Normalized', 'NDVI', 'LST_Delta_Ref'], description='Layer:', style=style)

reference_year_widget = widgets.Dropdown(
    options=reference_years,
    description='Anno Riferimento (Delta LST):',
    style=style,
    value=reference_years[-1] if reference_years else None
)

def update_map(year, month, layer_type, ref_year):
    for layer in Map.layers:
        if layer.name in ['Analisi Layer', 'Delta LST']:
            Map.remove_layer(layer)

    image = image_collection_dict.get(year, {}).get(month)

    if image:

        if layer_type == 'LST_Delta_Ref':
            reference_image = image_collection_dict.get(ref_year, {}).get(month)

            if reference_image:
                try:
                    current_lst = image.select('LST_Celsius')
                    reference_lst = reference_image.select('LST_Celsius')

                    lst_delta = current_lst.subtract(reference_lst).rename('LST_Delta')

                    max_delta = 5

                    delta_vis_params = {
                        'min': -max_delta,
                        'max': max_delta,
                        'palette': ['blue', 'cyan', 'yellow', 'orange', 'red'],
                        'forceRgbOutput': True
                    }

                    Map.add_layer(lst_delta, delta_vis_params, f'Delta LST ({year} vs {ref_year})')
                    print(f"Caricato: Mappa di Divergenza LST_Celsius (Delta {year} vs {ref_year}) per {month}")

                    return

                except Exception as e:
                    print(f"Errore nel calcolo/caricamento del Delta LST: {e}")
                    return
            else:
                print(f"Immagine di riferimento non disponibile per {month}/{ref_year}.")
                return

        else:
            vis_params = {}
            if layer_type == 'NDVI':
                vis_params = {'min': 0, 'max': 0.8, 'palette': ['blue', 'cyan', 'yellow', 'red']}
            elif 'LST' in layer_type:
                vis_params = {'min': 10, 'max': 45, 'palette': ['blue', 'cyan', 'yellow', 'red']}

            if image.bandNames().getInfo() and layer_type in image.bandNames().getInfo():
                try:
                    Map.add_layer(image.select(layer_type), vis_params, 'Analisi Layer')
                    print(f"Caricato: {layer_type} per {month}/{year}")
                except Exception as e:
                    print("Errore nel caricamento del layer.")
            else:
                print(f"Layer '{layer_type}' non trovato nell'immagine per {month}/{year}.")
    else:
        print("Immagine non disponibile.")

controls = widgets.VBox([
    year_widget,
    month_widget,
    layer_widget,
    reference_year_widget
])

output = widgets.interactive_output(
    update_map,
    {'year': year_widget, 'month': month_widget, 'layer_type': layer_widget, 'ref_year': reference_year_widget}
)

display(controls, output)

Map.add_layer(area, {}, 'Confine Genova')

Map

Output()

Map(center=[44.405, 8.934], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## 7. Analisi LST vs Land Cover Mese per Mese

Questa sezione modifica la logica di riduzione raggruppata per calcolare la media dell'LST (assoluta e normalizzata) e dell'NDVI per ogni classe di Land Cover (Land Cover) per ogni mese/anno della tua serie temporale.

1. Definizione della Funzione di Calcolo

Creiamo una funzione che calcola le statistiche raggruppate per una singola immagine, raggruppando i pixel per la loro classe Land Cover.

In [ ]:
wc = ee.ImageCollection("ESA/WorldCover/v200").first().clip(area)

lc_classes = {
    10: 'Alberi/Foresta',
    20: 'Arbusti',
    30: 'Prati/Erba',
    40: 'Coltivazioni',
    50: 'Costruito/Urbano',
    60: 'Vegetazione rada',
    80: 'Acqua',
    90: 'Zone Umide'
}

ts_data = []

for year in years:
    for month in months:
        img = image_collection_dict.get(year, {}).get(month)

        if img:
            combined = img.select(['LST_Normalized']).addBands(wc.rename('LandCover'))

            # calcolo le temperature medie per ogni zona
            try:
                stats = combined.reduceRegion(
                    reducer=ee.Reducer.mean().group(
                        groupField=1,
                        groupName='class_code'
                    ),
                    geometry=area,
                    scale=50,
                    maxPixels=1e9
                ).getInfo()

                date_val = pd.to_datetime(f'{year}-{month}-01')

                for item in stats.get('groups', []):
                    c_code = int(item['class_code'])
                    mean_temp = item['mean']

                    if c_code in lc_classes:
                        ts_data.append({
                            'Date': date_val,
                            'Year': year,
                            'Month': month,
                            'Class': lc_classes[c_code],
                            'LST_Normalized': mean_temp
                        })
            except Exception as e:
                pass

df_ts = pd.DataFrame(ts_data)

if not df_ts.empty:
    df_ts = df_ts.sort_values('Date')

    fig = px.line(df_ts, x='Date', y='LST_Normalized', color='Class',
                  title='Evoluzione Mensile della Temperatura Normalizzata per Copertura del Suolo',
                  labels={'LST_Normalized': 'Temperatura Normalizzata (°C)', 'Date': 'Data'},
                  markers=True,
                  color_discrete_map={
                      'Costruito/Urbano': 'red',
                      'Alberi/Foresta': 'green',
                      'Acqua': 'blue',
                      'Arbusti': 'orange',
                      'Prati/Erba': 'lightgreen'
                  })

    fig.update_layout(hovermode="x unified")
    fig.show()

    try:
        pivot_df = df_ts.pivot(index='Date', columns='Class', values='LST_Normalized')

        # La UHI Intensity è la differenza tra la temperatura urbana e quella forestale.
        pivot_df['UHI_Intensity'] = pivot_df['Costruito/Urbano'] - pivot_df['Alberi/Foresta']

        fig_uhi = px.bar(pivot_df, x=pivot_df.index, y='UHI_Intensity',
                         title='Intensità Isola di Calore (Differenza: Urbano - Foresta)',
                         labels={'UHI_Intensity': 'Delta T (°C)', 'Date': 'Data'},
                         color='UHI_Intensity',
                         color_continuous_scale='RdYlBu_r')
        fig_uhi.show()
    except KeyError:
        print("Dati insufficienti per calcolare la differenza Urbano-Foresta.")

else:
    print("Nessun dato estratto. Verifica che 'image_collection_dict' sia popolato e che i filtri siano corretti.")

# Mappa interattiva delle classi di copertura del suolo
Map_lc = geemap.Map(center=[44.405, 8.934], zoom=12)

lc_vis_params = {
    'min': 10,
    'max': 90,
    'palette': [
        '#008000',  # 10 Alberi/Foresta (Verde scuro)
        '#8B4513',  # 20 Arbusti (Marrone)
        '#90EE90',  # 30 Prati/Erba (Verde chiaro)
        '#FFD700',  # 40 Coltivazioni (Oro)
        '#B22222',  # 50 Costruito/Urbano (Rosso mattone)
        '#D3D3D3',  # 60 Vegetazione rada (Grigio chiaro)
        '#0000FF',  # 80 Acqua (Blu)
        '#ADD8E6'   # 90 Zone Umide (Azzurro)
    ]
}

Map_lc.add_layer(wc, lc_vis_params, 'Classi Land Cover')
Map_lc.add_layer(area, {}, 'Confine Genova')
Map_lc.add_legend(title="Classi Land Cover", labels=[lc_classes[c] for c in sorted(lc_classes.keys()) if c != 70], colors=[lc_vis_params['palette'][i] for i, c_code in enumerate(sorted(lc_classes.keys())) if c_code != 70])
Map_lc


ValueError: The legend keys and colors must be the same length.